In [1]:
#NAIVE RAG using Llama Index

# %pip install llama-index llama-index-llms-ollama llama-index-embeddings-ollama
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [2]:
#configuration
Settings.llm = Ollama(model="gemma4:e4b",
                    temperature=0.7,
                    num_predictions=300,
                    request_timeout=600)
Settings.embed_model = OllamaEmbedding(model_name="qwen3-embedding:0.6b")
Settings.chunk_size_limit = 512
Settings.chunk_overlap = 50

In [3]:
#document loading

from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader("data/").load_data()

In [4]:
#main

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex

# 1. Parse into nodes explicitly
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

# 2. Build the index from nodes (not from raw documents)
index = VectorStoreIndex(nodes)

# 3. Build retriever explicitly (this is the piece that changes across architectures)
retriever = index.as_retriever(similarity_top_k=5)

# 4. Build query engine from the retriever
from llama_index.core.query_engine import RetrieverQueryEngine
query_engine = RetrieverQueryEngine.from_args(retriever)

In [5]:
#Query

response = query_engine.query("summerize the introduction")
print(response.response)
print([n.text for n in response.source_nodes])  # this is your `contexts` field for RAGAS/DeepEval

The information details the architecture and implementation of a system that enhances Retrieval-Augmented Generation (RAG) by analyzing and decomposing non-factoid questions. This system, which employs various prompt templates, addresses several complex question types, including Debate, Experience, Reason, Instruction, and Comparison. For instance, debate-type questions are analyzed to extract the topic, identify diverse perspectives, and generate multiple biased sub-queries. Similarly, other question types are processed to transform the original query into focused sub-queries or keywords, ensuring that the retrieval process aligns with the user's specific intent and allows for a synthesized, accurate answer.
['Based on the query type, apply the relevant prompt to transform the query to better \r\n      align with the user’s intent, ensuring clarity and precision.\r\n      The input question is a debate-type question (i.e., invites multiple perspectives). As a "Query Analyst", please e